### Configuration Spark

In [6]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window

val spark = SparkSession.builder()
  .appName("Sales Analysis")
  .master("local[*]")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")  // pour réduire le bruit


spark = org.apache.spark.sql.SparkSession@311d2162


org.apache.spark.sql.SparkSession@311d2162

### Définir les schémas et charger les CSV

In [9]:
// --- Schema Products ---
val schemaProducts = StructType(Array(
  StructField("product_id", IntegerType, true),
  StructField("product_name", StringType, true),
  StructField("price", DoubleType, true)
))

// --- Schema Sales ---
val schemaSales = StructType(Array(
  StructField("order_id", IntegerType, true),
  StructField("product_id", IntegerType, true),
  StructField("seller_id", IntegerType, true),
  StructField("date", DateType, true) ,
  StructField("num_pieces_sold", IntegerType, true),
  StructField("bill_raw_text", StringType, true) ,

))

// --- Schema Sellers ---
val schemaSellers = StructType(Array(
  StructField("seller_id", IntegerType, true),
  StructField("seller_name", StringType, true),
  StructField("daily_target", IntegerType, true)
))

val dfProducts = spark.read.option("header", "true").schema(schemaProducts).csv("products.csv")
val dfSales = spark.read.option("header", "true").schema(schemaSales).csv("sales.csv")
val dfSellers = spark.read.option("header", "true").schema(schemaSellers).csv("sellers.csv")

// Afficher 5 premières lignes et schéma
dfProducts.show(5)
dfProducts.printSchema()

dfSales.show(5)
dfSales.printSchema()

dfSellers.show(5)
dfSellers.printSchema()


+----------+------------+-----+
|product_id|product_name|price|
+----------+------------+-----+
|         0|   product_0| 25.0|
|         1|   product_1| 30.0|
|         2|   product_2| 91.0|
|         3|   product_3| 37.0|
|         4|   product_4|145.0|
+----------+------------+-----+
only showing top 5 rows

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- price: double (nullable = true)

+--------+----------+---------+----------+---------------+--------------------+
|order_id|product_id|seller_id|      date|num_pieces_sold|       bill_raw_text|
+--------+----------+---------+----------+---------------+--------------------+
|       1|         0|        0|2022-07-03|             61|Apnkojtnlebiezbfd...|
|       2|         0|        0|2022-07-01|             69|wvuhynrzbtvjubjok...|
|       3|         0|        0|2022-07-02|             82|xsatoqobejrsbdkcb...|
|       4|         0|        0|2022-07-10|             79|jmisicdfqklgdcfdb..

schemaProducts = StructType(StructField(product_id,IntegerType,true), StructField(product_name,StringType,true), StructField(price,DoubleType,true))
schemaSales = StructType(StructField(order_id,IntegerType,true), StructField(product_id,IntegerType,true), StructField(seller_id,IntegerType,true), StructField(date,DateType,true), StructField(num_pieces_sold,IntegerType,true), StructField(bill_raw_text,StringType,true))
schemaSellers = StructType(StructField(seller_id,IntegerType,true), StructField(seller_name,StringType,true), StructField(daily_target,IntegerType,true))
dfProducts = [product_id: int, product_name: string ... 1 more field]


dfS...


[product_id: int, product_name: string ... 1 more field]

### Création des vues pour Spark SQL

In [10]:
dfProducts.createOrReplaceTempView("Products")
dfSales.createOrReplaceTempView("Sales")
dfSellers.createOrReplaceTempView("Sellers")


### Jointures et vues combinées

In [12]:
val salesWithProducts = spark.sql("""
  SELECT s.order_id, s.product_id, p.product_name, s.seller_id, s.num_pieces_sold, p.price, s.date
  FROM Sales s
  JOIN Products p
    ON s.product_id = p.product_id
""")
salesWithProducts.show(5)


+--------+----------+------------+---------+---------------+-----+----------+
|order_id|product_id|product_name|seller_id|num_pieces_sold|price|      date|
+--------+----------+------------+---------+---------------+-----+----------+
|   39989|         0|   product_0|        0|             67| 25.0|2022-07-01|
|   39988|         0|   product_0|        0|             71| 25.0|2022-07-07|
|   39987|         0|   product_0|        0|             34| 25.0|2022-07-03|
|   39986|         0|   product_0|        0|             72| 25.0|2022-07-07|
|   39985|         0|   product_0|        0|             71| 25.0|2022-07-03|
+--------+----------+------------+---------+---------------+-----+----------+
only showing top 5 rows



lastException = null
salesWithProducts = [order_id: int, product_id: int ... 5 more fields]


[order_id: int, product_id: int ... 5 more fields]

### Vue produit / prix / quantité vendue

In [14]:
val productSalesSummary = spark.sql("""
  SELECT p.product_name, p.price, SUM(s.num_pieces_sold) AS total_pieces_sold
  FROM Sales s
  JOIN Products p
    ON s.product_id = p.product_id
  GROUP BY p.product_name, p.price
""")
productSalesSummary.show()


+-------------+-----+-----------------+
| product_name|price|total_pieces_sold|
+-------------+-----+-----------------+
|  product_653|105.0|               50|
| product_2951| 54.0|               26|
| product_7940| 80.0|               11|
|product_13576|116.0|               86|
|product_22441|104.0|               64|
|product_28756| 62.0|               20|
|product_51925| 30.0|               62|
|product_60516|145.0|               77|
|product_10280|116.0|               29|
|product_12995|  7.0|               48|
|product_20755| 56.0|               35|
|product_21386|  4.0|               25|
|product_24624|147.0|               46|
|product_25925|142.0|                4|
|product_29218|  3.0|               14|
|product_38251| 82.0|               47|
|product_38597| 37.0|               21|
|product_40015| 76.0|               31|
|product_42680|  2.0|               32|
|product_44233|135.0|               85|
+-------------+-----+-----------------+
only showing top 20 rows



lastException = null
productSalesSummary = [product_name: string, price: double ... 1 more field]


[product_name: string, price: double ... 1 more field]

### Revenu moyen des commandes

In [15]:
val avgRevenue = salesWithProducts.withColumn("revenue", col("num_pieces_sold") * col("price"))
  .agg(avg("revenue").alias("avg_revenue"))
avgRevenue.show()


+------------------+
|       avg_revenue|
+------------------+
|1404.3094655344655|
+------------------+



avgRevenue = [avg_revenue: double]


[avg_revenue: double]

### Jointure Left Join sur les 3 tables

In [16]:
val fullView = dfSales
  .join(dfProducts, "product_id")
  .join(dfSellers, Seq("seller_id"), "left")
fullView.show(5)


+---------+----------+--------+----------+---------------+--------------------+------------+-----+-----------+------------+
|seller_id|product_id|order_id|      date|num_pieces_sold|       bill_raw_text|product_name|price|seller_name|daily_target|
+---------+----------+--------+----------+---------------+--------------------+------------+-----+-----------+------------+
|        0|         0|       1|2022-07-03|             61|Apnkojtnlebiezbfd...|   product_0| 25.0|   seller_0|     2500000|
|        0|         0|       2|2022-07-01|             69|wvuhynrzbtvjubjok...|   product_0| 25.0|   seller_0|     2500000|
|        0|         0|       3|2022-07-02|             82|xsatoqobejrsbdkcb...|   product_0| 25.0|   seller_0|     2500000|
|        0|         0|       4|2022-07-10|             79|jmisicdfqklgdcfdb...|   product_0| 25.0|   seller_0|     2500000|
|        0|         0|       5|2022-07-09|             52|lnisjbcxswqkloycq...|   product_0| 25.0|   seller_0|     2500000|
+-------

fullView = [seller_id: int, product_id: int ... 8 more fields]


[seller_id: int, product_id: int ... 8 more fields]

### Comparer les performances des vendeurs

In [17]:
val sellerPerformance = fullView.withColumn("revenue", col("num_pieces_sold") * col("price"))
  .groupBy("seller_id", "seller_name", "daily_target")
  .agg(sum("revenue").alias("total_sales"))
  .withColumn("performance_vs_target", col("total_sales") - col("daily_target"))
sellerPerformance.show()


+---------+-----------+------------+-----------+---------------------+
|seller_id|seller_name|daily_target|total_sales|performance_vs_target|
+---------+-----------+------------+-----------+---------------------+
|        0|   seller_0|     2500000|   4.8281E7|             4.5781E7|
|        8|   seller_8|     1528179|   958296.0|            -569883.0|
|        3|   seller_3|     1157798|   958747.0|            -199051.0|
|        9|   seller_9|      442654|   748737.0|             306083.0|
|        4|   seller_4|     1999213|   819888.0|           -1179325.0|
|        7|   seller_7|      888438|   924374.0|              35936.0|
|        2|   seller_2|      117965|   964732.0|             846767.0|
|        6|   seller_6|      499726|   744715.0|             244989.0|
|        1|   seller_1|      988094|   908569.0|             -79525.0|
|        5|   seller_5|     1793991|   919493.0|            -874498.0|
+---------+-----------+------------+-----------+---------------------+



sellerPerformance = [seller_id: int, seller_name: string ... 3 more fields]


[seller_id: int, seller_name: string ... 3 more fields]

### Fonctions Window Avancées

In [19]:
// Classement des vendeurs par chiffre d'affaires
val windowBySeller = Window.orderBy(desc("total_sales"))
val rankedSellers = sellerPerformance.withColumn("rank", rank().over(windowBySeller))
rankedSellers.show()




+---------+-----------+------------+-----------+---------------------+----+
|seller_id|seller_name|daily_target|total_sales|performance_vs_target|rank|
+---------+-----------+------------+-----------+---------------------+----+
|        0|   seller_0|     2500000|   4.8281E7|             4.5781E7|   1|
|        2|   seller_2|      117965|   964732.0|             846767.0|   2|
|        3|   seller_3|     1157798|   958747.0|            -199051.0|   3|
|        8|   seller_8|     1528179|   958296.0|            -569883.0|   4|
|        7|   seller_7|      888438|   924374.0|              35936.0|   5|
|        5|   seller_5|     1793991|   919493.0|            -874498.0|   6|
|        1|   seller_1|      988094|   908569.0|             -79525.0|   7|
|        4|   seller_4|     1999213|   819888.0|           -1179325.0|   8|
|        9|   seller_9|      442654|   748737.0|             306083.0|   9|
|        6|   seller_6|      499726|   744715.0|             244989.0|  10|
+---------+-

lastException = null
windowBySeller = org.apache.spark.sql.expressions.WindowSpec@f4b7041
rankedSellers = [seller_id: int, seller_name: string ... 4 more fields]


[seller_id: int, seller_name: string ... 4 more fields]

In [20]:
// Montant cumulé des ventes par vendeur au fil du temps
val cumulativeSales = fullView.withColumn("revenue", col("num_pieces_sold") * col("price"))
  .withColumn("cumulative_sales", sum("revenue").over(Window.partitionBy("seller_id").orderBy("date")))
cumulativeSales.show()



+---------+----------+--------+----------+---------------+--------------------+-------------+-----+-----------+------------+-------+----------------+
|seller_id|product_id|order_id|      date|num_pieces_sold|       bill_raw_text| product_name|price|seller_name|daily_target|revenue|cumulative_sales|
+---------+----------+--------+----------+---------------+--------------------+-------------+-----+-----------+------------+-------+----------------+
|        1|     19601|    2982|2022-07-01|             12|tjiacdbfxxQlzodrp...|product_19601| 14.0|   seller_1|      988094|  168.0|        112967.0|
|        1|      1222|    2996|2022-07-01|              8|fzooukoosiejkweoe...| product_1222|129.0|   seller_1|      988094| 1032.0|        112967.0|
|        1|     36643|    7988|2022-07-01|             14|luehnehgbNffvibiy...|product_36643|127.0|   seller_1|      988094| 1778.0|        112967.0|
|        1|     56061|    7998|2022-07-01|             38|mfjfgiwrqumeizcbv...|product_56061|116.0| 

cumulativeSales = [seller_id: int, product_id: int ... 10 more fields]


[seller_id: int, product_id: int ... 10 more fields]

In [21]:
// Comparer chaque vente à la moyenne du produit
val windowByProduct = Window.partitionBy("product_id")
val salesVsAvg = fullView.withColumn("revenue", col("num_pieces_sold") * col("price"))
  .withColumn("avg_product_sales", avg("revenue").over(windowByProduct))
salesVsAvg.show()



+---------+----------+--------+----------+---------------+--------------------+-------------+-----+-----------+------------+-------+-----------------+
|seller_id|product_id|order_id|      date|num_pieces_sold|       bill_raw_text| product_name|price|seller_name|daily_target|revenue|avg_product_sales|
+---------+----------+--------+----------+---------------+--------------------+-------------+-----+-----------+------------+-------+-----------------+
|        3|      3749|   38034|2022-07-07|             21|imcddekahjzkohimm...| product_3749| 84.0|   seller_3|     1157798| 1764.0|           1764.0|
|        3|      8086|   40020|2022-07-01|             31|vhiouaxssmnysknvv...| product_8086|132.0|   seller_3|     1157798| 4092.0|           4092.0|
|        5|     13289|   20006|2022-07-07|             89|zazuDvxoprovriigi...|product_13289| 33.0|   seller_5|     1793991| 2937.0|           2937.0|
|        4|     14450|   12994|2022-07-05|             61|lxwmrltklkfjvzccc...|product_14450|1

windowByProduct = org.apache.spark.sql.expressions.WindowSpec@3d11c9f3
salesVsAvg = [seller_id: int, product_id: int ... 10 more fields]


[seller_id: int, product_id: int ... 10 more fields]

### Créer les case classes

In [23]:
import java.sql.Date

case class Product(product_id: Int, product_name: String, price: Double)
case class Sale(order_id: Int, product_id: Int, seller_id: Int, date: Date, num_pieces_sold: Int, bill_raw_text: String)
case class Seller(seller_id: Int, seller_name: String, daily_target: Int)


lastException = null
defined class Product
defined class Sale
defined class Seller


null

### Convertir les DataFrames en Datasets typés

In [24]:
import spark.implicits._

val dsProducts = dfProducts.as[Product]
val dsSales = dfSales.as[Sale]
val dsSellers = dfSellers.as[Seller]


dsProducts = [product_id: int, product_name: string ... 1 more field]
dsSales = [order_id: int, product_id: int ... 4 more fields]
dsSellers = [seller_id: int, seller_name: string ... 1 more field]


[seller_id: int, seller_name: string ... 1 more field]

### Déterminer les produits les plus vendus

In [25]:
val topProducts = dsSales.join(dsProducts, "product_id")
  .groupBy("product_name")
  .agg(sum("num_pieces_sold").alias("total_sold"))
  .orderBy(desc("total_sold"))

topProducts.show()


+-------------+----------+
| product_name|total_sold|
+-------------+----------+
|    product_0|   1931240|
|product_57727|       179|
|product_28161|       168|
|product_40106|       142|
|product_29541|       138|
| product_6068|       136|
|product_17998|       131|
|product_55851|       125|
|product_30413|       114|
|product_15565|       113|
|product_66599|       112|
|product_18575|       105|
|product_13529|       104|
|product_58800|       102|
|product_63569|       102|
|product_23445|       101|
|product_36516|       100|
|product_15837|       100|
|product_35484|       100|
|product_55222|       100|
+-------------+----------+
only showing top 20 rows



topProducts = [product_name: string, total_sold: bigint]


[product_name: string, total_sold: bigint]

### Analyser les performances des vendeurs

In [26]:
val sellerPerformance = dsSales
  .join(dsProducts, "product_id")
  .join(dsSellers, "seller_id")
  .groupBy("seller_name", "daily_target")
  .agg(sum(col("num_pieces_sold") * col("price")).alias("total_sales"))
  .withColumn("performance_gap", col("total_sales") - col("daily_target"))

sellerPerformance.show()


+-----------+------------+-----------+---------------+
|seller_name|daily_target|total_sales|performance_gap|
+-----------+------------+-----------+---------------+
|   seller_1|      988094|   908569.0|       -79525.0|
|   seller_5|     1793991|   919493.0|      -874498.0|
|   seller_4|     1999213|   819888.0|     -1179325.0|
|   seller_3|     1157798|   958747.0|      -199051.0|
|   seller_8|     1528179|   958296.0|      -569883.0|
|   seller_2|      117965|   964732.0|       846767.0|
|   seller_6|      499726|   744715.0|       244989.0|
|   seller_0|     2500000|   4.8281E7|       4.5781E7|
|   seller_7|      888438|   924374.0|        35936.0|
|   seller_9|      442654|   748737.0|       306083.0|
+-----------+------------+-----------+---------------+



sellerPerformance = [seller_name: string, daily_target: int ... 2 more fields]


[seller_name: string, daily_target: int ... 2 more fields]

### Analyser les ventes par catégorie (avec sous-performances)

In [28]:
import org.apache.spark.sql.functions._

// --- Exemple : ajout d'une colonne fictive 'category' dans dsProducts ---
val dsProductsWithCategory = dsProducts.withColumn("category", lit("Electronics"))

// --- Ventes par catégorie ---
val categorySales = dsSales
  .join(dsProductsWithCategory, "product_id")
  .groupBy("category")
  .agg(sum(col("num_pieces_sold") * col("price")).alias("total_revenue"))

// --- Détection des catégories sous-performantes (objectif fixe 20%) ---
val fixedTarget = 0.2 * 10000  // 20% d'un objectif fixe de 10000
val categoryAnalysisFixed = categorySales.withColumn(
  "underperforming_fixed",
  col("total_revenue") < lit(fixedTarget)   // lit() autour de la constante
)

// --- Objectif dynamique : comparer au revenu moyen de toutes les catégories ---
val avgRevenue = categorySales.agg(avg("total_revenue").alias("avg_category_revenue")).first().getDouble(0)
val categoryAnalysisDynamic = categorySales.withColumn(
  "underperforming_dynamic",
  col("total_revenue") < lit(0.8 * avgRevenue)   // lit() autour de la constante
)

// --- Affichage des résultats ---
categoryAnalysisFixed.show()
categoryAnalysisDynamic.show()


+-----------+-------------+---------------------+
|   category|total_revenue|underperforming_fixed|
+-----------+-------------+---------------------+
|Electronics|  5.6228551E7|                false|
+-----------+-------------+---------------------+

+-----------+-------------+-----------------------+
|   category|total_revenue|underperforming_dynamic|
+-----------+-------------+-----------------------+
|Electronics|  5.6228551E7|                  false|
+-----------+-------------+-----------------------+



dsProductsWithCategory = [product_id: int, product_name: string ... 2 more fields]
categorySales = [category: string, total_revenue: double]
fixedTarget = 2000.0
categoryAnalysisFixed = [category: string, total_revenue: double ... 1 more field]
avgRevenue = 5.6228551E7
categoryAnalysisDynamic = [category: string, total_revenue: double ... 1 more field]


[category: string, total_revenue: double ... 1 more field]

### Comparaison DataFrame vs Dataset

### Version DataFrame (DF)

In [29]:
import org.apache.spark.sql.functions._
import org.apache.spark.sql.DataFrame
import org.apache.spark.util.LongAccumulator
import scala.concurrent.duration._

val t0 = System.nanoTime()  // début du chrono

val dfResult = dfSales
  .join(dfProducts, "product_id")
  .groupBy("seller_id")
  .agg(sum(col("num_pieces_sold") * col("price")).as("revenue"))
  .count()  // Force l'exécution

val t1 = System.nanoTime()  // fin du chrono
println(s"DataFrame execution time: ${(t1 - t0)/1e9} seconds")


DataFrame execution time: 2.117590684 seconds


t0 = 2141138253590
dfResult = 10
t1 = 2143255844274


2143255844274

### Version Dataset (DS)

In [30]:
import spark.implicits._
import org.apache.spark.sql.expressions.scalalang.typed

val t0_ds = System.nanoTime()

val dsResult = dsSales
  .joinWith(dsProducts, dsSales("product_id") === dsProducts("product_id"))
  .map { case (sale, product) => (sale.seller_id, sale.num_pieces_sold * product.price) }
  .groupByKey(_._1)
  .agg(typed.sum(_._2))
  .count()  // Même opération finale

val t1_ds = System.nanoTime()
println(s"Dataset execution time: ${(t1_ds - t0_ds)/1e9} seconds")


Dataset execution time: 2.843186764 seconds


t0_ds = 2158322495948
dsResult = 10
t1_ds = 2161165682712


2161165682712

### Analyse et mesure des performances

Pour analyser nombre de jobs/stages, volume shuffle, mémoire et serialization/deserialization, on peut utiliser :

1- Spark UI → http://localhost:4040 pendant l’exécution pour :

- Jobs / Stages

- Shuffle read/write

- Memory usage

- Serialization / Deserialization

2- Quelques métriques via Spark SQL (optionnel) :

In [32]:
// DataFrame
val t0 = System.nanoTime()
val dfCount = dfSales
  .join(dfProducts, "product_id")
  .groupBy("seller_id")
  .agg(sum(col("num_pieces_sold") * col("price")).as("revenue"))
  .count()
val t1 = System.nanoTime()
println(s"DataFrame count: $dfCount, execution time: ${(t1-t0)/1e9} seconds")

// Dataset
val t0_ds = System.nanoTime()
val dsCount = dsSales
  .joinWith(dsProducts, dsSales("product_id") === dsProducts("product_id"))
  .map { case (sale, product) => (sale.seller_id, sale.num_pieces_sold * product.price) }
  .groupByKey(_._1)
  .agg(typed.sum(_._2))
  .count()
val t1_ds = System.nanoTime()
println(s"Dataset count: $dsCount, execution time: ${(t1_ds-t0_ds)/1e9} seconds")


DataFrame count: 10, execution time: 1.787884532 seconds
Dataset count: 10, execution time: 1.859137407 seconds


t0 = 2363698485444
dfCount = 10
t1 = 2365486369976
t0_ds = 2365486577874
dsCount = 10
t1_ds = 2367345715281


2367345715281

### Conseils pour l’optimisation

- Partitionnement : Utiliser repartition() ou coalesce() pour équilibrer les partitions.

- Caching : df.cache() ou ds.cache() si le dataset est réutilisé plusieurs fois.

- Projection / filtrage tôt : Sélectionner seulement les colonnes nécessaires avant les joins.

- Dataset API : Moins de shuffle si tu utilises joinWith et map typé, comparé à DataFrame avec expressions SQL.

### Résumé attendu :

- Dataset (DS) = souvent plus rapide et moins de shuffle si opérations typées et transformations map/filter.

- DataFrame (DF) = plus flexible avec Spark SQL, mais peut générer plus de shuffle et overhead si le plan d’exécution est complexe.